# OList E-commerce — Regression Track (Review 1)

## Objective

Predict the continuous target variable **`price`** using the OList e-commerce dataset.

This notebook is organized to satisfy **Review 1, Section C — Regression Track (9 marks)**:

- **C1:** Train all 10 required regression algorithms.
- **C2:** Compare all models using R², RMSE, and MAE on the same held-out test set.
- **C3:** Apply hyperparameter tuning to at least two models and report the improvement.
- **C4:** Produce a residual plot, predicted-vs-actual plot, and tree-based feature-importance plot.

The notebook also includes dataset auditing, EDA, written observation prompts, data cleaning, feature engineering, leakage-safe preprocessing, and 5-fold cross-validation for the two best-performing models.

> **Reproducibility:** `random_state=42` is used wherever it is applicable.

## Important modeling decisions

1. `price` is the regression target because it is continuous.
2. Duplicate rows are removed.
3. The timestamp is converted into meaningful date/time features.
4. Identifier and free-text columns are removed because they are high-cardinality or difficult to use directly in this baseline pipeline.
5. Missing values are handled inside a scikit-learn preprocessing pipeline.
6. Encoders and scalers are fitted only on the training data to avoid data leakage.
7. All standard models use the same processed train/test split.
8. Polynomial regression uses a controlled subset of numerical transformed features to prevent a very large polynomial feature matrix.
9. SVR is trained on a capped, reproducible training subset because kernel SVR can become computationally expensive on large datasets. It is still evaluated on the same held-out test set.

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
    cross_val_score
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "price"

print("Libraries imported successfully.")

In [ ]:
# ============================================================
# 2. LOAD THE DATASET
# ============================================================

# The first path supports a notebook running beside the CSV file.
# The second path supports the uploaded-file environment.
possible_paths = [
    Path("OList-Ecommerce.csv"),
    Path("data/OList-Ecommerce.csv"),
    Path("../data/OList-Ecommerce.csv"),
    Path("/mnt/data/OList-Ecommerce.csv")
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "OList-Ecommerce.csv was not found. "
        "Place the CSV in the notebook directory or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")
display(df.head())


In [ ]:
# ============================================================
# 3. DATASET AUDIT
# ============================================================

print("Data types and non-null counts:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("\nMissing-value summary:")
display(missing_summary[missing_summary["missing_count"] > 0])

duplicate_count = df.duplicated().sum()
print(f"Number of completely duplicated rows: {duplicate_count}")
print(f"Duplicate percentage: {(duplicate_count / len(df) * 100):.2f}%")

print(f"Missing values in target '{TARGET}': {df[TARGET].isna().sum()}")

### Dataset audit — written observation

The audit reports the dataset dimensions, data types, missing-value counts, duplicate count, and target completeness.

When presenting this section, explain:

- Whether the target contains missing values.
- Which columns have the most missing values.
- Why duplicate rows should be removed before model training.
- Why identifier and free-text columns require special treatment.

In [ ]:
# ============================================================
# 4. TARGET DISTRIBUTION
# ============================================================

plt.figure(figsize=(9, 5))
sns.histplot(df[TARGET].dropna(), kde=True)
plt.title("Distribution of Product Price")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Target statistics:")
display(df[TARGET].describe())
print(f"Target skewness: {df[TARGET].skew():.3f}")

### Target-distribution observation

Use the displayed histogram and skewness value to comment on:

- Whether prices are approximately symmetric or right-skewed.
- Whether a small number of high-priced products may influence the regression models.
- Why MAE and RMSE are both reported: RMSE gives more weight to large errors.

In [ ]:
# ============================================================
# 5. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

numeric_columns_initial = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns_initial = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:")
print(numeric_columns_initial)

print("\nCategorical/object columns:")
print(categorical_columns_initial)

In [ ]:
# ============================================================
# 6. NUMERICAL FEATURE DISTRIBUTIONS
# ============================================================

# Plot every numerical feature separately so that each feature
# can be inspected without overlapping scales.
for column in numeric_columns_initial:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[column].dropna(), kde=True)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

### Numerical-distribution observation

For each numerical feature, identify visible concentration, skewness, or extreme values.

Do not automatically delete every extreme value. In an e-commerce dataset, high prices, large freight values, and large product weights may be legitimate observations.

In [ ]:
# ============================================================
# 7. CATEGORICAL FEATURE DISTRIBUTIONS
# ============================================================

# Count plots are shown only for low-cardinality columns.
# High-cardinality columns are summarized through value counts.
for column in categorical_columns_initial:
    unique_count = df[column].nunique(dropna=False)

    if unique_count <= 20:
        plt.figure(figsize=(8, 4))
        order = df[column].fillna("Missing").value_counts().index
        sns.countplot(
            data=df.assign(**{column: df[column].fillna("Missing")}),
            x=column,
            order=order
        )
        plt.title(f"Distribution of {column}")
        plt.xlabel(column)
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"{column}: {unique_count} unique values")
        display(df[column].value_counts(dropna=False).head(10))

In [ ]:
# ============================================================
# 8. CORRELATION HEATMAP
# ============================================================

numeric_for_corr = df.select_dtypes(include=np.number)

plt.figure(figsize=(10, 7))
sns.heatmap(
    numeric_for_corr.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

target_correlations = (
    numeric_for_corr.corr()[TARGET]
    .drop(TARGET)
    .sort_values(key=np.abs, ascending=False)
)

print("Numerical feature correlations with price:")
display(target_correlations.to_frame("correlation"))

### Correlation-heatmap observation

Use the heatmap and correlation table to discuss:

- Which numerical features have the strongest positive or negative linear relationship with `price`.
- Whether any numerical predictors are strongly correlated with each other.
- Why correlation does not prove causation and may not capture non-linear relationships.

In [ ]:
# ============================================================
# 9. TWO FEATURE-TARGET SCATTER PLOTS
# ============================================================

# Select up to two numerical features with the strongest absolute
# correlation with the target. This makes the scatter plots data-driven.
top_scatter_features = target_correlations.abs().head(2).index.tolist()

for feature in top_scatter_features:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=df,
        x=feature,
        y=TARGET,
        alpha=0.45
    )
    plt.title(f"{feature} vs {TARGET}")
    plt.xlabel(feature)
    plt.ylabel(TARGET)
    plt.tight_layout()
    plt.show()

print("Features selected for target scatter plots:", top_scatter_features)

### Scatter-plot observation

For each scatter plot, comment on:

- Whether the relationship appears positive, negative, weak, or non-linear.
- Whether the points contain visible clusters or extreme observations.
- Whether the plot supports using a linear model alone or motivates testing non-linear models.

In [ ]:
# ============================================================
# 10. REMOVE DUPLICATES AND HANDLE THE TARGET
# ============================================================

rows_before = len(df)

# Remove exact duplicate records.
df = df.drop_duplicates().reset_index(drop=True)

# Rows without a target cannot be used for supervised regression.
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print(f"Rows before cleaning: {rows_before}")
print(f"Rows after duplicate removal and target cleaning: {len(df)}")
print(f"Remaining missing target values: {df[TARGET].isna().sum()}")

In [ ]:
# ============================================================
# 11. TIMESTAMP FEATURE ENGINEERING
# ============================================================

# Convert the timestamp safely. Invalid timestamps become NaT.
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

# Extract useful calendar/time information.
df["purchase_year"] = df["order_purchase_timestamp"].dt.year
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_day"] = df["order_purchase_timestamp"].dt.day
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek

print("Engineered timestamp features:")
print([
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_hour",
    "purchase_dayofweek"
])

### Feature-engineering justification

The timestamp is transformed into year, month, day, hour, and day-of-week features because purchasing patterns can vary by season, month, time of day, and weekday.

The original timestamp is removed later because most scikit-learn estimators cannot directly use a raw datetime column in this pipeline.

In [ ]:
# ============================================================
# 12. OUTLIER AUDIT USING THE IQR METHOD
# ============================================================

# The IQR method is used for auditing, not for blindly deleting
# legitimate high-value products.
numeric_columns_after_engineering = df.select_dtypes(include=np.number).columns.tolist()

outlier_rows = []

for column in numeric_columns_after_engineering:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (df[column] < lower_bound) | (df[column] > upper_bound)
    count = int(outlier_mask.sum())

    outlier_rows.append({
        "feature": column,
        "outlier_count": count,
        "outlier_percentage": round(count / len(df) * 100, 2),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values(
    "outlier_count", ascending=False
)

display(outlier_summary)

### Outlier-treatment decision

The IQR output identifies potential outliers, but the project retains observations unless there is evidence that they are invalid.

This is appropriate for e-commerce data because unusually expensive products or large freight values may be genuine business records. The models are evaluated with MAE and RMSE so that the effect of large errors can be observed.

In [ ]:
# ============================================================
# 13. DEFINE FEATURES AND REMOVE UNSUITABLE COLUMNS
# ============================================================

# Separate target and predictors.
X = df.drop(columns=[TARGET])
y = df[TARGET].copy()

# These columns are removed because they are:
# - unique identifiers,
# - free-text review fields, or
# - raw datetime values that are replaced by engineered features.
columns_to_drop = [
    "order_id",
    "customer_id",
    "product_id",
    "seller_id",
    "review_comment_title",
    "review_comment_message",
    "order_purchase_timestamp"
]

X = X.drop(columns=columns_to_drop, errors="ignore")

print("Final raw feature columns:")
print(X.columns.tolist())
print(f"Feature matrix shape before splitting: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# ============================================================
# 14. TRAIN-TEST SPLIT
# ============================================================

# Use one fixed split for fair comparison of all models.
# Stratification is not appropriate for continuous regression targets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

In [ ]:
# ============================================================
# 15. LEAKAGE-SAFE PREPROCESSING PIPELINE
# ============================================================

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Numerical columns:
# 1. Median imputation handles missing numerical values.
# 2. StandardScaler is important for Ridge, Lasso, ElasticNet,
#    SVR, KNN, and polynomial regression.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical columns:
# 1. Most-frequent imputation fills missing categories.
# 2. One-hot encoding converts categories into numerical columns.
# 3. min_frequency reduces extremely rare categories and helps
#    control the number of generated columns.
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )
except TypeError:
    # Compatibility with older scikit-learn versions.
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", one_hot_encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Fit only on training data; then transform both training and test data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

### Preprocessing justification

The preprocessing object is fitted only on `X_train`.

This prevents data leakage because:

- Imputation values are learned from the training data.
- Scaling parameters are learned from the training data.
- Categorical levels are learned from the training data.
- The test set is transformed using the already-fitted preprocessing object.

In [ ]:
# ============================================================
# 16. PREPARE DATA FOR MODELS
# ============================================================

# Some estimators accept sparse matrices, while others require
# dense arrays. The dataset is kept in a dense representation
# for consistent use across the required regression algorithms.
X_train_model = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else np.asarray(X_train_processed)
X_test_model = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else np.asarray(X_test_processed)

print("Dense training matrix shape:", X_train_model.shape)
print("Dense testing matrix shape:", X_test_model.shape)

# Polynomial regression is applied to a controlled subset of the
# numerical transformed columns. This avoids an uncontrolled
# explosion in the number of polynomial features.
poly_feature_count = min(5, len(numeric_features))
X_train_poly_base = X_train_model[:, :poly_feature_count]
X_test_poly_base = X_test_model[:, :poly_feature_count]

print("Number of numerical features used for polynomial expansion:", poly_feature_count)

In [ ]:
# ============================================================
# 17. EVALUATION FUNCTION
# ============================================================

def calculate_regression_metrics(y_true, y_pred):
    """Return the required Review 1 regression metrics."""
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

def evaluate_and_store(model_name, model, X_fit, y_fit, X_eval, y_eval, results, predictions):
    """Fit a model, generate predictions, and store metrics."""
    model.fit(X_fit, y_fit)
    prediction = model.predict(X_eval)

    metrics = calculate_regression_metrics(y_eval, prediction)
    metrics["Model"] = model_name

    results.append(metrics)
    predictions[model_name] = prediction

    print(f"{model_name} completed.")
    print(
        f"R2={metrics['R2']:.4f}, "
        f"RMSE={metrics['RMSE']:.4f}, "
        f"MAE={metrics['MAE']:.4f}"
    )

    return model

# Section C1 — Implementation of all 10 regression algorithms

All required algorithms are trained below. The same held-out test set is used for evaluation.

The polynomial model uses the controlled numerical subset described earlier. SVR uses a reproducible training subset to keep kernel-based computation practical for this dataset size.

In [ ]:
# ============================================================
# 18. TRAIN THE 10 REQUIRED REGRESSION MODELS
# ============================================================

results = []
predictions = {}
trained_models = {}

# 1. Linear Regression — baseline model.
trained_models["Linear Regression"] = evaluate_and_store(
    "Linear Regression",
    LinearRegression(),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 2. Ridge Regression — L2 regularization.
trained_models["Ridge Regression"] = evaluate_and_store(
    "Ridge Regression",
    Ridge(alpha=1.0),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 3. Lasso Regression — L1 regularization.
trained_models["Lasso Regression"] = evaluate_and_store(
    "Lasso Regression",
    Lasso(alpha=0.001, max_iter=5000),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 4. ElasticNet — combined L1 and L2 regularization.
trained_models["ElasticNet Regression"] = evaluate_and_store(
    "ElasticNet Regression",
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 5. Polynomial Regression.
# PolynomialFeatures is applied to a controlled numerical subset.
poly_model = Pipeline(steps=[
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear_regression", LinearRegression())
])

trained_models["Polynomial Regression"] = evaluate_and_store(
    "Polynomial Regression",
    poly_model,
    X_train_poly_base, y_train,
    X_test_poly_base, y_test,
    results, predictions
)


In [ ]:

# 6. Decision Tree Regressor.
trained_models["Decision Tree Regressor"] = evaluate_and_store(
    "Decision Tree Regressor",
    DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE
    ),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 7. Random Forest Regressor.
trained_models["Random Forest Regressor"] = evaluate_and_store(
    "Random Forest Regressor",
    RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 8. Gradient Boosting Regressor.
trained_models["Gradient Boosting Regressor"] = evaluate_and_store(
    "Gradient Boosting Regressor",
    GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)

# 9. Support Vector Regressor.
# Kernel SVR can be expensive for large datasets, so a fixed
# subset is used for fitting while the complete test set is used.
svr_train_size = min(8000, len(X_train_model))
svr_rng = np.random.RandomState(RANDOM_STATE)
svr_indices = svr_rng.choice(len(X_train_model), size=svr_train_size, replace=False)

trained_models["Support Vector Regressor"] = evaluate_and_store(
    "Support Vector Regressor",
    SVR(C=10.0, kernel="rbf", epsilon=0.1),
    X_train_model[svr_indices], y_train.iloc[svr_indices],
    X_test_model, y_test,
    results, predictions
)

# 10. K-Nearest Neighbors Regressor.
trained_models["K-Nearest Neighbors Regressor"] = evaluate_and_store(
    "K-Nearest Neighbors Regressor",
    KNeighborsRegressor(n_neighbors=5, weights="distance"),
    X_train_model, y_train,
    X_test_model, y_test,
    results, predictions
)
print("\nAll 10 required regression algorithms have been processed.")


In [ ]:
# ============================================================
# 19. C2 — COMPARATIVE EVALUATION TABLE
# ============================================================

comparison_table = pd.DataFrame(results)

# Rank by R2 in descending order, as required by the rubric.
comparison_table = comparison_table.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

comparison_table = comparison_table[
    ["Model", "R2", "RMSE", "MAE"]
]

display(comparison_table.style.format({
    "R2": "{:.4f}",
    "RMSE": "{:.4f}",
    "MAE": "{:.4f}"
}))

print("The table compares all 10 models on the same held-out test set.")

### Comparative evaluation observation

Use the comparison table to explain:

- Which model has the highest test R².
- Which model has the lowest RMSE and MAE.
- Whether the model with the highest R² also has the lowest error values.
- Why model selection should consider multiple metrics rather than only one score.

# Section C3 — Hyperparameter tuning

At least two models are tuned using `GridSearchCV`.

The tuning section reports:

- The search space.
- The best parameter combination.
- The baseline test metric.
- The tuned test metric.
- The change in R².

In [ ]:
# ============================================================
# 20. C3 — HYPERPARAMETER TUNING: RIDGE
# ============================================================

ridge_grid = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge_search = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_grid,
    scoring="r2",
    cv=3,
    n_jobs=-1
)

ridge_search.fit(X_train_model, y_train)
ridge_tuned_predictions = ridge_search.best_estimator_.predict(X_test_model)
ridge_tuned_metrics = calculate_regression_metrics(y_test, ridge_tuned_predictions)

ridge_baseline_r2 = comparison_table.loc[
    comparison_table["Model"] == "Ridge Regression", "R2"
].iloc[0]

print("Ridge best parameters:", ridge_search.best_params_)
print(f"Ridge baseline R2: {ridge_baseline_r2:.4f}")
print(f"Ridge tuned R2: {ridge_tuned_metrics['R2']:.4f}")
print(f"Ridge R2 improvement: {ridge_tuned_metrics['R2'] - ridge_baseline_r2:.4f}")

In [ ]:
# ============================================================
# 21. C3 — HYPERPARAMETER TUNING: RANDOM FOREST
# ============================================================

random_forest_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2]
}

random_forest_search = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_grid=random_forest_grid,
    scoring="r2",
    cv=3,
    n_jobs=-1
)

random_forest_search.fit(X_train_model, y_train)
rf_tuned_predictions = random_forest_search.best_estimator_.predict(X_test_model)
rf_tuned_metrics = calculate_regression_metrics(y_test, rf_tuned_predictions)

rf_baseline_r2 = comparison_table.loc[
    comparison_table["Model"] == "Random Forest Regressor", "R2"
].iloc[0]

print("Random Forest best parameters:", random_forest_search.best_params_)
print(f"Random Forest baseline R2: {rf_baseline_r2:.4f}")
print(f"Random Forest tuned R2: {rf_tuned_metrics['R2']:.4f}")
print(f"Random Forest R2 improvement: {rf_tuned_metrics['R2'] - rf_baseline_r2:.4f}")

In [ ]:
# ============================================================
# 22. TUNING SUMMARY TABLE
# ============================================================

tuning_summary = pd.DataFrame([
    {
        "Model": "Ridge Regression",
        "Baseline R2": ridge_baseline_r2,
        "Tuned R2": ridge_tuned_metrics["R2"],
        "R2 Improvement": ridge_tuned_metrics["R2"] - ridge_baseline_r2,
        "Best Parameters": str(ridge_search.best_params_)
    },
    {
        "Model": "Random Forest Regressor",
        "Baseline R2": rf_baseline_r2,
        "Tuned R2": rf_tuned_metrics["R2"],
        "R2 Improvement": rf_tuned_metrics["R2"] - rf_baseline_r2,
        "Best Parameters": str(random_forest_search.best_params_)
    }
])

display(tuning_summary.style.format({
    "Baseline R2": "{:.4f}",
    "Tuned R2": "{:.4f}",
    "R2 Improvement": "{:.4f}"
}))

### Hyperparameter-tuning observation

The tuning summary should be discussed using the actual printed results.

Mention:

- Which parameters were selected.
- Whether the tuned model improved the test R².
- Whether the improvement is large, small, or negative.
- Why tuning results should be interpreted carefully because the test set is used only for final evaluation.

# 5-fold cross-validation for the two best-performing models

The two models are selected from the baseline comparison table. Cross-validation is performed on the training data only, using R² as the scoring metric.

In [ ]:
# ============================================================
# 23. 5-FOLD CROSS-VALIDATION FOR THE TWO BEST MODELS
# ============================================================

# Select the two best baseline models by test R2.
top_two_models = comparison_table["Model"].head(2).tolist()
print("Two models selected for 5-fold cross-validation:", top_two_models)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

for model_name in top_two_models:
    if model_name == "Linear Regression":
        cv_model = LinearRegression()
        cv_X = X_train_model
    elif model_name == "Ridge Regression":
        cv_model = Ridge(alpha=1.0)
        cv_X = X_train_model
    elif model_name == "Lasso Regression":
        cv_model = Lasso(alpha=0.001, max_iter=5000)
        cv_X = X_train_model
    elif model_name == "ElasticNet Regression":
        cv_model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000)
        cv_X = X_train_model
    elif model_name == "Polynomial Regression":
        cv_model = Pipeline(steps=[
            ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
            ("linear_regression", LinearRegression())
        ])
        cv_X = X_train_poly_base
    elif model_name == "Decision Tree Regressor":
        cv_model = DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=RANDOM_STATE)
        cv_X = X_train_model
    elif model_name == "Random Forest Regressor":
        cv_model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
        cv_X = X_train_model
    elif model_name == "Gradient Boosting Regressor":
        cv_model = GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)
        cv_X = X_train_model
    elif model_name == "Support Vector Regressor":
        cv_model = SVR(C=10.0, kernel="rbf", epsilon=0.1)
        cv_X = X_train_model[svr_indices]
        y_cv = y_train.iloc[svr_indices]
        scores = cross_val_score(cv_model, cv_X, y_cv, cv=cv, scoring="r2", n_jobs=-1)
        cv_results.append({
            "Model": model_name,
            "Mean CV R2": scores.mean(),
            "Std CV R2": scores.std()
        })
        continue
    elif model_name == "K-Nearest Neighbors Regressor":
        cv_model = KNeighborsRegressor(n_neighbors=5, weights="distance")
        cv_X = X_train_model
    else:
        raise ValueError(f"Unknown model: {model_name}")

    scores = cross_val_score(
        cv_model,
        cv_X,
        y_train,
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "Mean CV R2": scores.mean(),
        "Std CV R2": scores.std()
    })

cv_summary = pd.DataFrame(cv_results)
display(cv_summary.style.format({
    "Mean CV R2": "{:.4f}",
    "Std CV R2": "{:.4f}"
}))

### Cross-validation observation

Explain the mean and standard deviation of the 5-fold R² scores.

- A higher mean CV R² indicates stronger average validation performance.
- A large standard deviation indicates that performance changes more across folds.
- The test-set score and cross-validation score may differ because they measure performance on different partitions.

# Section C4 — Required regression visualizations

The following cells produce the required plots for the best baseline model selected by test R².

In [ ]:
# ============================================================
# 24. SELECT THE BEST BASELINE MODEL
# ============================================================

best_model_name = comparison_table.iloc[0]["Model"]
best_predictions = predictions[best_model_name]

print("Best baseline model according to test R2:", best_model_name)
print("Best model metrics:")
display(comparison_table.iloc[[0]])

In [ ]:
# ============================================================
# 25. C4 — PREDICTED VS ACTUAL PLOT
# ============================================================

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=y_test,
    y=best_predictions,
    alpha=0.55
)

minimum_value = min(y_test.min(), best_predictions.min())
maximum_value = max(y_test.max(), best_predictions.max())

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
    label="Perfect prediction"
)

plt.title(f"Predicted vs Actual Price — {best_model_name}")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 26. C4 — RESIDUAL PLOT
# ============================================================

residuals = y_test.to_numpy() - best_predictions

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=best_predictions,
    y=residuals,
    alpha=0.55
)
plt.axhline(0, linestyle="--")
plt.title(f"Residual Plot — {best_model_name}")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.4f}")
print(f"Residual standard deviation: {residuals.std():.4f}")

In [ ]:
# ============================================================
# 27. C4 — TREE-BASED FEATURE IMPORTANCE
# ============================================================

# Random Forest is used for the required feature-importance plot.
# The feature names are obtained from the fitted preprocessing object.
feature_names = preprocessor.get_feature_names_out()

rf_model_for_importance = trained_models["Random Forest Regressor"]
rf_importances = rf_model_for_importance.feature_importances_

importance_table = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_importances
}).sort_values("Importance", ascending=False)

top_importances = importance_table.head(20).sort_values("Importance")

plt.figure(figsize=(10, 8))
plt.barh(top_importances["Feature"], top_importances["Importance"])
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(importance_table.head(20))

### Visualization observations

For the final presentation, explain:

1. **Predicted vs actual plot:** Points close to the diagonal indicate more accurate predictions.
2. **Residual plot:** Look for systematic patterns, unequal spread, or clusters rather than assuming that all residuals are random.
3. **Feature importance:** The plot shows which encoded features the Random Forest used most strongly. Feature importance is not proof of causation.

In [ ]:
# ============================================================
# 28. OPTIONAL: SAVE THE FINAL COMPARISON TABLES
# ============================================================

comparison_table.to_csv("regression_comparison_results.csv", index=False)
tuning_summary.to_csv("regression_tuning_summary.csv", index=False)
cv_summary.to_csv("regression_cross_validation_results.csv", index=False)
importance_table.to_csv("random_forest_feature_importance.csv", index=False)

print("Result tables saved successfully.")